In [1]:
# ============================================
# PHASE 1: LOAD AND UNDERSTAND THE DATA
# Store Item Demand Forecasting
# ============================================
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# --------------------------------------------
# 1. Load the dataset
# --------------------------------------------

# Change this path to wherever your train.csv is located
file_path = "train.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!\n")


# --------------------------------------------
# 2. Basic information
# --------------------------------------------

print("Shape of dataset:")
print(df.shape)

print("\nFirst 5 rows:")
print(df.head())


# --------------------------------------------
# 3. Check data types
# --------------------------------------------

print("\nData types:")
print(df.dtypes)


# --------------------------------------------
# 4. Convert date column to datetime
# --------------------------------------------

df["date"] = pd.to_datetime(df["date"])

print("\nDate range:")
print("Start:", df["date"].min())
print("End:  ", df["date"].max())


# --------------------------------------------
# 5. Check missing values
# --------------------------------------------

print("\nMissing values:")
print(df.isnull().sum())


# --------------------------------------------
# 6. Check duplicate rows
# --------------------------------------------

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())


# --------------------------------------------
# 7. Check unique stores and items
# --------------------------------------------

print("\nNumber of stores:", df["store"].nunique())
print("Number of items:", df["item"].nunique())


print("\nStores:")
print(sorted(df["store"].unique()))

print("\nItems:")
print(sorted(df["item"].unique()))


# --------------------------------------------
# 8. Check observations per store-item pair
# --------------------------------------------

store_item_counts = (
    df.groupby(["store", "item"])
      .size()
      .reset_index(name="observations")
)

print("\nObservations per Store-Item combination:")
print(store_item_counts["observations"].describe())


# --------------------------------------------
# 9. Sort the data
# --------------------------------------------

df = df.sort_values(
    by=["store", "item", "date"]
).reset_index(drop=True)


# --------------------------------------------
# 10. Verify the sorted data
# --------------------------------------------

print("\nSorted data:")
print(df.head(10))


# --------------------------------------------
# 11. Basic sales statistics
# --------------------------------------------

print("\nSales statistics:")
print(df["sales"].describe())


# --------------------------------------------
# 12. Verify each Store-Item combination
# --------------------------------------------

print("\nNumber of Store × Item combinations:")
print(df.groupby(["store", "item"]).ngroups)


# --------------------------------------------
# 13. Final dataset information
# --------------------------------------------

print("\nFinal dataset information:")
df.info()

In [2]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df["day_name"] = df["date"].dt.day_name()
df['DayOfWeek'] = df['date'].dt.dayofweek

In [3]:
daily_sales = (
    df.groupby("date")["sales"]
      .sum()
      .reset_index()
)

print(daily_sales.head())

In [4]:


plt.figure(figsize=(15, 5))

plt.plot(
    daily_sales["date"],
    daily_sales["sales"]
)

plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.title("Overall Daily Sales")

plt.show()

In [5]:
weekly_sales = (
    df.groupby("day_name")["sales"]
      .mean()
      .reindex([
          "Monday",
          "Tuesday",
          "Wednesday",
          "Thursday",
          "Friday",
          "Saturday",
          "Sunday"
      ])
)

print(weekly_sales)
plt.figure(figsize=(10, 5))

weekly_sales.plot(kind="bar")

plt.xlabel("Day of Week")
plt.ylabel("Average Sales")
plt.title("Average Sales by Day of Week")

plt.show()

In [6]:
monthly_sales = (
    df.groupby("month")["sales"]
      .mean()
)

print(monthly_sales)
plt.figure(figsize=(10, 5))

monthly_sales.plot(kind="bar")

plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.title("Average Sales by Month")

plt.show()

In [7]:
store_sales = (
    df.groupby("store")["sales"]
      .mean()
      .sort_values(ascending=False)
)

print(store_sales)
plt.figure(figsize=(10, 5))

store_sales.plot(kind="bar")

plt.xlabel("Store")
plt.ylabel("Average Daily Sales")
plt.title("Average Sales by Store")

plt.show()

In [8]:
item_sales = (
    
    df.groupby("item")["sales"]
      .mean()
      .sort_values(ascending=False)
)

print(item_sales.head(10))
plt.figure(figsize=(12, 5))

item_sales.head(10).plot(kind="bar")

plt.xlabel("Item")
plt.ylabel("Average Daily Sales")
plt.title("Top 10 Items by Average Sales")

plt.show()

In [9]:
store_item_sales = (
    df.groupby(["store", "item"])["sales"]
      .mean()
      .reset_index()
      .sort_values("sales", ascending=False)
)

print(store_item_sales.head(10))


In [10]:
item_volatility = (
    df.groupby("item")["sales"]
      .std()
      .sort_values(ascending=False)
)

print(item_volatility.head(10))

In [11]:
store_volatility = (
    df.groupby("store")["sales"]
      .std()
      .sort_values(ascending=False)
)

print(store_volatility)

In [12]:
plt.figure(figsize=(10, 5))

plt.hist(df["sales"], bins=50)

plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.title("Distribution of Daily Sales")

plt.show()

In [13]:
import matplotlib.pyplot as plt

# Calculate average sales for each item
item_sales = df.groupby('item')['sales'].mean().sort_values(ascending=False)

# Plot
plt.figure(figsize=(12, 6))
plt.bar(item_sales.index, item_sales.values)

plt.title('Average Sales by Item')
plt.xlabel('Item')
plt.ylabel('Average Sales')
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

In [14]:
df['Lag_1'] = df.groupby(['store', 'item'])['sales'].shift(1)
df['Lag_7'] = df.groupby(['store', 'item'])['sales'].shift(7)
df['Lag_14'] = df.groupby(['store', 'item'])['sales'].shift(14)
df['Lag_28'] = df.groupby(['store', 'item'])['sales'].shift(28)


In [15]:
df['Rolling_Mean_7'] = (
    df.groupby(['store', 'item'])['sales']
      .transform(lambda x: x.shift(1).rolling(7).mean())
)
df['Rolling_Mean_14'] = (
    df.groupby(['store', 'item'])['sales']
      .transform(lambda x: x.shift(1).rolling(14).mean())
)
df['Rolling_Mean_28'] = (
    df.groupby(['store', 'item'])['sales']
      .transform(lambda x: x.shift(1).rolling(28).mean())
)

In [16]:
df['Rolling_Std_7'] = (
    df.groupby(['store', 'item'])['sales']
      .transform(lambda x: x.shift(1).rolling(7).std())
)

In [17]:
df['Sales_Change_1'] = (
    df['Lag_1'] - df['Lag_7']
)
df['Sales_Growth_7'] = (
    (df['Lag_1'] - df['Lag_7']) /
    (df['Lag_7'] + 1)
)

In [18]:
df.head(35)

In [19]:
df = df.dropna()

In [20]:
print(df.head())
print(df.columns)
df['WeekOfYear'] = df['date'].dt.isocalendar().week.astype(int)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype('category')

In [21]:

cutoff_date = df['date'].max() - pd.Timedelta(days=90)

train = df[df['date'] <= cutoff_date]
test = df[df['date'] > cutoff_date]

In [22]:
print("Train:", train['date'].min(), "to", train['date'].max())
print("Test :", test['date'].min(), "to", test['date'].max())

In [23]:
y_train = train['sales']
y_test = test['sales']

In [24]:
features = [
    'store',
    'item',
    'year',
    'month',
    'day_name',
    'day_of_week',
    'WeekOfYear',
    'Lag_1',
    'Lag_7',
    'Lag_14',
    'Lag_28',
    'Rolling_Mean_7',
    'Rolling_Mean_14',
    'Rolling_Mean_28',
    'Rolling_Std_7',
    'Sales_Change_1',
    'Sales_Growth_7'
]

X_train = train[features]
X_test = test[features]

In [25]:
print("Last training date:", train['date'].max())
print("First test date:", test['date'].min())

In [26]:
print(train['date'].max() < test['date'].min())

In [27]:



from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    enable_categorical=True,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

model.fit(
    X_train,
    y_train
)

In [28]:
y_pred = model.predict(X_test)

In [29]:
print(y_pred[:10])

In [30]:
results = test[['date', 'store', 'item', 'sales']].copy()

results['predicted_sales'] = y_pred

print(results.head(20))

In [31]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)

In [32]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)

In [33]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred)

print("MAPE:", mape * 100)

In [34]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    results['date'],
    results['sales'],
    label='Actual Sales'
)
plt.plot(
    results['date'],
    results['predicted_sales'],
    label='Predicted Sales'
)



plt.title('Actual vs Predicted Sales')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()

plt.tight_layout()
plt.show()

In [35]:
baseline_pred = test['Lag_7']

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

print("Baseline MAE:", baseline_mae)

In [36]:
store_id = 1
item_id = 1

plot_data = results[
    (results['store'] == store_id) &
    (results['item'] == item_id)
]

plt.figure(figsize=(14, 6))

plt.plot(
    plot_data['date'],
    plot_data['sales'],
    label='Actual'
)

plt.plot(
    plot_data['date'],
    plot_data['predicted_sales'],
    label='Predicted'
)

plt.title(f'Actual vs Predicted Sales - Store {store_id}, Item {item_id}')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()

plt.tight_layout()
plt.show()

In [37]:
import matplotlib.pyplot as plt

importance = model.feature_importances_

plt.figure(figsize=(10, 6))

plt.barh(features, importance)

plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('XGBoost Feature Importance')

plt.tight_layout()
plt.show()

In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Training predictions
y_train_pred = model.predict(X_train)

# Training metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

# Test metrics
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Training MAE :", train_mae)
print("Training RMSE:", train_rmse)

print("Test MAE     :", test_mae)
print("Test RMSE    :", test_rmse)

In [39]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

tuned_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

tuned_model.fit(X_train, y_train)

# Predictions
tuned_pred = tuned_model.predict(X_test)

# Evaluation
tuned_mae = mean_absolute_error(y_test, tuned_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_pred))

print("Tuned XGBoost MAE :", tuned_mae)
print("Tuned XGBoost RMSE:", tuned_rmse)

In [40]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(y_test.values, label='Actual')
plt.plot(tuned_pred, label='Predicted')

plt.title('Actual vs Predicted Sales')
plt.xlabel('Test Observations')
plt.ylabel('Sales')
plt.legend()
plt.show()

In [41]:
correlation = np.corrcoef(y_test, tuned_pred)[0, 1]

print("Actual-Predicted Correlation:", correlation)

In [42]:
bias = np.mean(tuned_pred - y_test)

print("Prediction Bias:", bias)

In [43]:
print(df.dtypes)

In [44]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Start with the most recent historical data
# --------------------------------------------------

history = df[['date', 'store', 'item', 'sales']].copy()

history = history.sort_values(
    ['store', 'item', 'date']
).reset_index(drop=True)

# --------------------------------------------------
# 2. Define forecast horizon
# --------------------------------------------------

forecast_days = 30

last_date = history['date'].max()

future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=forecast_days,
    freq='D'
)

# --------------------------------------------------
# 3. Get all store-item combinations
# --------------------------------------------------

store_items = history[['store', 'item']].drop_duplicates()

future = (
    store_items.assign(key=1)
    .merge(
        pd.DataFrame({'date': future_dates, 'key': 1}),
        on='key'
    )
    .drop(columns='key')
)

future['sales'] = np.nan

# --------------------------------------------------
# 4. Combine historical + future rows
# --------------------------------------------------

forecast_df = pd.concat(
    [history, future],
    ignore_index=True
)

forecast_df = forecast_df.sort_values(
    ['store', 'item', 'date']
).reset_index(drop=True)


forecast_df["day_name"] = forecast_df.date.dt.day_name()
forecast_df['day_name'] = forecast_df['day_name'].astype('category')
# --------------------------------------------------
# 5. Recursive forecasting
# --------------------------------------------------

# forecast_df['WeekOfYear'] = forecast_df['date'].dt.isocalendar().week.astype(int)


for current_date in future_dates:

    mask = forecast_df['date'] == current_date

    # Calendar features
    forecast_df.loc[mask, 'year'] = current_date.year
    
    forecast_df.loc[mask, 'month'] = current_date.month
    forecast_df.loc[mask, 'day_of_week'] = current_date.dayofweek
    forecast_df.loc[mask, 'WeekOfYear'] = current_date.isocalendar().week

    # Calculate lag features using values available
    forecast_df['Lag_1'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .shift(1)
    )

    forecast_df['Lag_7'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .shift(7)
    )

    forecast_df['Lag_14'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .shift(14)
    )

    forecast_df['Lag_28'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .shift(28)
    )

    # Rolling features
    forecast_df['Rolling_Mean_7'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .transform(
            lambda x: x.shift(1).rolling(7).mean()
        )
    )

    forecast_df['Rolling_Mean_14'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .transform(
            lambda x: x.shift(1).rolling(14).mean()
        )
    )

    forecast_df['Rolling_Mean_28'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .transform(
            lambda x: x.shift(1).rolling(28).mean()
        )
    )

    forecast_df['Rolling_Std_7'] = (
        forecast_df.groupby(['store', 'item'])['sales']
        .transform(
            lambda x: x.shift(1).rolling(7).std()
        )
    )
    
    forecast_df['Sales_Change_1'] = (
     forecast_df['Lag_1'] -  forecast_df['Lag_7']
)

    forecast_df['Sales_Growth_7'] = (
    (forecast_df['Lag_1'] - forecast_df['Lag_7']) /
    (forecast_df['Lag_7'] + 1)
)


    # Select current future rows
    current_rows = forecast_df.loc[mask, features]

    # Predict
    predictions = tuned_model.predict(current_rows)

    # Store predictions as sales
    forecast_df.loc[mask, 'sales'] = predictions

# --------------------------------------------------
# 6. Extract final forecasts
# --------------------------------------------------

future_forecast = forecast_df[
    forecast_df['date'] > last_date
].copy()

print(future_forecast[
    ['date', 'store', 'item', 'sales']
].head(20))

In [45]:
# Overall forecast statistics

print("Forecast period:")
print(future_forecast['date'].min(), "to", future_forecast['date'].max())

print("\nTotal forecasted sales:")
print(future_forecast['sales'].sum())

print("\nAverage daily forecasted sales:")
print(future_forecast.groupby('date')['sales'].sum().mean())

print("\nMinimum daily forecasted sales:")
print(future_forecast.groupby('date')['sales'].sum().min())

print("\nMaximum daily forecasted sales:")
print(future_forecast.groupby('date')['sales'].sum().max())

In [46]:
# Overall forecast statistics

print("Forecast period:")
print(future_forecast['date'].min(), "to", future_forecast['date'].max())

print("\nTotal forecasted sales:")
print(future_forecast['sales'].sum())

print("\nAverage daily forecasted sales:")
print(future_forecast.groupby('date')['sales'].sum().mean())

print("\nMinimum daily forecasted sales:")
print(future_forecast.groupby('date')['sales'].sum().min())

print("\nMaximum daily forecasted sales:")
print(future_forecast.groupby('date')['sales'].sum().max())

In [47]:
store_forecast = (
    future_forecast
    .groupby('store')['sales']
    .sum()
    .sort_values(ascending=False)
)

print(store_forecast)

In [48]:
import matplotlib.pyplot as plt

store_forecast.plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title("Forecasted Sales by Store")
plt.xlabel("Store")
plt.ylabel("Forecasted Sales")
plt.tight_layout()
plt.show()

In [49]:
item_forecast = (
    future_forecast
    .groupby('item')['sales']
    .sum()
    .sort_values(ascending=False)
)

print(item_forecast)
item_forecast.plot(
    kind='bar',
    figsize=(12, 5)
)

plt.title("Forecasted Sales by Item")
plt.xlabel("Item")
plt.ylabel("Forecasted Sales")
plt.tight_layout()
plt.show()

In [50]:
# Last 90 days of actual sales
df['date'] = pd.to_datetime(df['date'])
cutoff_date = df['date'].max() - pd.Timedelta(days=90)

last_90_days = df[
    df['date'] > cutoff_date
].copy()

print("Last 90 days:")
print(last_90_days['date'].min())
print(last_90_days['date'].max())

historical_daily = (
    last_90_days
    .groupby('date')['sales']
    .sum()
    .reset_index()
)
daily_forecast = (
    future_forecast
    .groupby('date')['sales']
    .sum()
    .reset_index()
)

print(daily_forecast.head())
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    historical_daily['date'],
    historical_daily['sales'],
    label='Actual Sales'
)

plt.plot(
    daily_forecast['date'],
    daily_forecast['sales'],
    label='Future Forecast',
    linewidth=2
)

plt.axvline(
    x=df['date'].max(),
    linestyle='--',
    label='Forecast Start'
)

plt.title('Last 90 Days Actual Sales + 30-Day Future Forecast')
plt.xlabel('Date')
plt.ylabel('Total Sales')

plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [51]:
from sklearn.metrics import mean_absolute_error

test_results = test[['date', 'store', 'item', 'sales']].copy()

test_results['prediction'] = tuned_pred

store_errors = (
    test_results
    .groupby('store')
    .apply(
        lambda x: mean_absolute_error(
            x['sales'],
            x['prediction']
        )
    )
    .sort_values()
)

print(store_errors)

In [52]:
item_errors = (
    test_results
    .groupby('item')
    .apply(
        lambda x: mean_absolute_error(
            x['sales'],
            x['prediction']
        )
    )
    .sort_values()
)

print(item_errors)

In [53]:
print("Best predicted items:")
print(item_errors.head(10))

In [54]:
print("Worst predicted items:")
print(item_errors.tail(10))

In [55]:
negative_forecasts = future_forecast[
    future_forecast['sales'] < 0
]

print("Negative predictions:", len(negative_forecasts))